## **一、S8-01｜读懂 Chimera，并决定什么值得带回 MiniGen**

### **1. 最后一课要回答的问题**

S4 解决“模型学什么速度场、怎样积分”；S5 解决“文本与图像怎样一起计算”；S6 解决“模态分支什么时候合并”；S7 解决“怎样可靠地训练、评测和解释结果”。最后一课继续追问：**当 token 越来越多时，是否每一层都值得让每个 token 直接查询全部 token？**

本课以 [Chimera v1](https://arxiv.org/html/2607.28611v1) 为研究入口，核实日期为 2026-11-08。重点是读懂注意力的计算与记忆方式，建立可检验的架构判断。课件中的完整参考只覆盖两个小算子；不训练图像模型，不下载权重，不创建正式 Chimera 分支。工程计划中的 B0/B1/B2 仍是主线，Chimera-Lite 是否开发仍由第一轮结果与后续授权决定。

前置笔记：[S4-02 潜空间 Flow](S4_02_LatentFlow与PlainDiT训练闭环.ipynb)、[S5-01 Joint Attention 与 RoPE](S5_01_JointAttention与2DRoPE.ipynb)、[S5-02 Single-Stream](S5_02_SingleStreamDiT架构与实现.ipynb)、[S6-01 Hybrid](S6_01_HybridDiT双流单流架构.ipynb)、[S7 训练与评测](S7_01-02_训练工程与可信评测实操.ipynb)。

| 本课层次 | 要掌握什么 | 本课提供的证据 |
| --- | --- | --- |
| 必须理解 | 全注意力、有限状态、KV 压缩分别保存什么 | 推导、维度表、手算、反例 |
| 亲手练习 | 先衰减、再纠错的一步状态更新 | 独立 TODO 与测试入口 |
| 参考验证 | KDA 递推与 MLA 线性吸收 | CPU 双精度输出、梯度、边界检查 |
| 研究判断 | 256px 是否值得增加架构复杂度 | 操作数分析与未来实测协议 |
| 未执行 | CUDA 性能、真实 Chimera、正式训练 | 不将理论估算记成实测结果 |

### **2. 先算一个实际问题：MiniGen 的序列有多长**

MiniGen 的 256×256 图像经 SDXL VAE 空间压缩 8 倍，得到 32×32 的潜变量；采用 2×2 latent patch，则图像有 16×16=256 个 token。文本最多 77 个位置，保留 padding 的固定长度上限为 333。若移除 padding，实际序列更短，但样本边界和 mask 也必须随之改变。

先运行下面的算术单元，比较几种序列长度。它计算的是 **显式 attention score 张量** 的元素数，以及一份 KDA 递推状态的大小；没有测延迟或训练显存。公式采用一个样本、一个层，8 个头，每头 key/value 维度均为 64；它们只是便于理解的配置，不是 Chimera 或 MiniGen 的正式参数。

In [1]:
from __future__ import annotations

import math

import torch
from jaxtyping import Float
from torch import Tensor

# 只计算张量容量：不是实测 CUDA 显存或延迟。
heads, head_width = 8, 64
state_mib = heads * head_width * head_width * 4 / 2**20
print(f"PyTorch {torch.__version__}; CPU; KDA 单层状态 {state_mib:.3f} MiB")
for length in (333, 1024, 4096):
    score_mib = heads * length**2 * 2 / 2**20
    print(f"n={length:4d}: 显式 BF16 score={score_mib:8.3f} MiB")
print(f"论文正文给定 FLOPs 的比值 = {4.29e20 / 6.27e19:.6f}")

PyTorch 2.14.1; CPU; KDA 单层状态 0.125 MiB
n= 333: 显式 BF16 score=   1.692 MiB
n=1024: 显式 BF16 score=  16.000 MiB
n=4096: 显式 BF16 score= 256.000 MiB
论文正文给定 FLOPs 的比值 = 6.842105


对 333 个位置，显式 BF16 分数矩阵约 1.69 MiB，一份 FP32 KDA 状态约 0.125 MiB；但 FlashAttention/融合 SDPA 本来就不需要完整保存那张分数矩阵。这个对比揭示数学增长趋势，**不能据此宣布训练显存下降约 13.5 倍**。参数、梯度、优化器、激活、反向重算以及其余层都未计入。

### **3. 先分清三种“流”和两种时间**

| 名词 | 在本课程中的意思 | 不应混淆为 |
| --- | --- | --- |
| Single / Dual Stream | 图文是否保留独立参数分支与表示路径 | 线性／全局注意力的选择 |
| Hybrid Attention | 不同层使用不同注意力算子 | 必然先双流再单流 |
| iHC residual streams | 每个 token 的多份残差表示 | 文本流、图像流或多张图片 |
| 扩散时间 $\tau$ | 噪声到数据路径上的连续位置 | token 下标 |
| 扫描位置 $j$ | 一次去噪前向中的第几个 token | 第几次 ODE 求值 |

一次 ODE 求值会调用整个去噪主干；主干中的每个 KDA 层又扫描一遍 token。进入下一个扩散时间后，图像 token 已变化，不能沿用上一轮的视觉递推状态来假装“继续扫描”。

## **二、架构地图｜把论文放回已经学过的系统**

### **1. 四种设计实际改变的是哪条轴**

| 架构 | 图文信息组织 | 层内主要交互 | 本课程的关注点 |
| --- | --- | --- | --- |
| B0 PlainConditionalDiT | 图像主干，文本作条件 | 图像 self-attention + text cross-attention | 最容易定位问题的基线 |
| B1 SingleStreamDiT | 投影后放入统一 token 序列 | Joint Attention | 条件与图像共享计算空间 |
| B2 HybridDiT | 前段双流、后段单流 | 全局图文交互，分支组织变化 | 模态专用变换与共享变换的取舍 |
| Chimera 研究参考 | 文本前缀接视觉序列 | 多数 KDA、周期性 MLA | 状态记忆与直接检索的取舍 |

Qwen 与 Hunyuan 的细节以前两课固定来源为准；表格是比较维度，不表示所有同名模型的实现完全一致。Chimera 的混合算子可以与“统一 token 序列”同时成立，因此它不与 S5/S6 的分类冲突。

### **2. 从输入到输出：什么改变，什么保持不变**

按 [Chimera §3](https://arxiv.org/html/2607.28611v1#S3) 的结构描述，冻结视觉编码器和文本编码器后，用不同输入投影把两种 token 映射到同一宽度。去噪块串行应用 attention 与 FFN，时间条件控制调制和残差门；最后仅视觉位置预测 Flow 速度。论文的视觉编码器、文本编码器及视频输入设置与 MiniGen 不同，不能直接复用本项目缓存来加载其权重。

本课用下面的 **MiniGen 尺度** 解释张量流；这里只是形状映射，不是已落地的新模型：

| 环节 | 张量 | 形状／含义 |
| --- | --- | --- |
| 图像 | $x$ | `b 3 256 256`，依组件要求预处理 |
| 缩放后的潜变量 | $z$ | `b 4 32 32`，缩放取组件配置 |
| 文本特征 | $c$ | `b n_t 512`，CLIP-B/32 冻结输出及有效 mask |
| 路径点 | $z_\tau$ | `b 4 32 32` |
| 视觉 patch | $p$ | `b 256 16`，16=4×2×2 |
| 共同隐藏序列 | $X$ | `b (n_t+256) d` |
| 可选 iHC 状态 | $R$ | `b (n_t+256) m d`，m 是残差副本数 |
| 视觉输出 | $\hat u$ | unpatchify 回 `b 4 32 32` |

图文使用不同输入投影，不表示后续一定保留双流；多个残差副本也不表示模态重新拆分。论文的 iHC、稀疏专家及性能内核不在本课参考实现中。

### **3. 目标函数仍与 S4 相连**

令 $z$ 为固定表示空间中的干净潜变量，$\epsilon\sim\mathcal N(0,I)$，取噪声为 $\tau=0$、数据为 $\tau=1$：

$$z_\tau=(1-\tau)\epsilon+\tau z,\qquad u=z-\epsilon$$

$$\mathcal L=\mathbb E_{z,c,\epsilon,\tau}\left[\frac{1}{c_z h_z w_z}\lVert v_\theta(z_\tau,\tau,c)-u\rVert_2^2\right]$$

这里每个样本在视觉元素上取均值，再对 batch 取均值；不同形状不能因元素多而无意获得更高权重。网络在一个给定路径点只观察 $z_\tau,\tau,c$，没有直接看到正确的 $z$ 和 $\epsilon$，因此需要学习条件速度。参考核的状态更新公式不替代这个训练目标。

训练时随机取路径点；生成时从噪声开始，按 Euler/Heun 多次调用速度模型。KDA 不会自动减少 NFE，也不把 Flow 变为自回归图像生成；采样预算仍按 S7 记账。

### **4. 对照原版配置，避免把教学尺寸当成论文模型**

以下是 [Chimera §5.1](https://arxiv.org/html/2607.28611v1#S5.SS1) 与附录B表4列明的配置。它们用于阅读定位，不是这份Notebook加载过的模型：

| 部分 | Chimera原版 | MiniGen／本课边界 |
| --- | --- | --- |
| 冻结表示 | Wan causal 3D VAE，16通道，时间／空间压缩4/8/8 | MiniGen SDXL VAE，4通道，二维；不得互换缓存 |
| 归一化 | 逐通道latent mean/std标准化 | MiniGen遵循SDXL组件自己的缩放协议 |
| patch | 时间／高／宽为1/2/2 | 单256px图同为256个视觉token，但原版每patch64维，本项目16维 |
| 文本 | 冻结umT5-XXL，上限512token，去padding后打包 | 本项目CLIP-B/32上限77；参考数值不调用任一文本塔 |
| 最终主干 | hidden2048、FFN8192、32层；激活2.22B／总11.28B | 摘要11B／2B是取整；本课只验证小算子 |
| 层与残差 | KDA:MLA=3:1，iHC为4个残差副本 | 对32层对应24个KDA与8个MLA；没有在本项目新增主干 |

时间压缩要处理第一帧：输入 $4k+1$ 帧得到 $k+1$ 个latent帧，不是简单对所有帧数整除4。81帧得到21个latent帧；单图对应 $k=0$。全文未找到足以确认最终模型head数量、head维度和MLA绝对rank的明确数值，本课不按2048的宽度自行猜测。

论文训练使用每样本一个扩散时间，所有token共享。它从logit-normal采样，再按视觉token数做shift；“loss不额外按时间加权”与“时间均匀采样”是两件不同的事。logit-normal一般指 $a\sim\mathcal N(\mu,\sigma^2)$、$\tau=\operatorname{sigmoid}(a)$；该段未列出确切 $\mu,\sigma$，不补造官方数值。

$$s=\sqrt{n_{\mathrm{vis}}/256},\qquad\tau'=\frac{\tau}{\tau+s(1-\tau)}$$

> 256px单图有256视觉token，所以 $s=1$，$\tau'=\tau$；1024px单图有4096视觉token，$s=4$。若原来 $\tau=0.5$，则 $\tau'=0.5/(0.5+4\times0.5)=0.2$。由于0是噪声端，变小意味着更多噪声；文本token数不计入这里的 $n_{\mathrm{vis}}$。

这是 **改变训练时间的抽样分布**：将 $\tau'$ 送入同一条线性路径与模型，速度目标仍是 $z-\epsilon$。它不是把ODE坐标重定义为原始抽样变量再做换元，不能无故把目标乘上 $d\tau'/d\tau$。如果要比较有无shift，必须重新固定验证时间分布；不能直接把不同目标权重下的loss大小当质量结论。

原训练配方还使用0.1条件丢弃、AdamW基础LR $10^{-3}$、betas=(0.95,0.95)、基础WD=0.01、基础epsilon=$10^{-8}$；2000步warmup后保持基础LR常数。参数组的实际数值再经HeteroP处理，不能把“base LR”当成所有参数的实际LR。EMA decay为0.9999；全局梯度裁剪阈值1.0，pre-clip norm非有限或大于10时跳过optimizer和EMA更新。跳步计数、成功更新计数与学习率日程必须有明确约定，不能删除这些样本后宣称完全稳定。

原系统还采用多分辨率图像／视频、HSDP、packed内核与负载均衡。它们解释了为什么原训练工程复杂，却不是MiniGen应照搬的默认配置。正式主线仍使用S7预登记协议；本课没有改时间分布、优化器或数据来源。

## **三、KDA 的数学｜一张会纠错的关联记忆**

### **1. 为什么用矩阵保存过去**

全注意力给当前 query 保留了逐项查询历史 key/value 的机会；记忆长度随序列增长。另一种思路是维护固定形状矩阵 $S\in\mathbb R^{d_k\times d_v}$：给它 key $k$，希望通过 $S^\top k$ 取回对应 value。这里把 key 看作列向量，所以读取结果是 $d_v$ 维。

最简单的写入是累加 $kv^\top$。但重复写入同一 key 可能让数值越积越大，而不是替换旧值；相似 key 也会相互干扰。**Delta rule** 的想法是先看当前记忆猜错了多少，再针对这个误差写入。它来自可直接求导的局部平方误差，不需要先记住一整套神经网络名词。[Kimi Linear §2–3](https://arxiv.org/html/2510.26692v1#S2) 给出这一视角。

### **2. 从局部平方误差推到状态更新**

先对旧状态各 key 通道衰减：

$$\bar S_j=A_jS_{j-1},\qquad A_j=\operatorname{Diag}(\alpha_j)$$

$\alpha_j\in(0,1)^{d_k}$ 控制每一行旧记忆保留多少；每一行对应 key 空间的一个方向，不是 value 通道。此时旧记忆对新 key 的预测为 $\hat v_j=\bar S_j^\top k_j$。定义暂时把 $k_j,v_j$ 固定的局部目标：

$$\ell_j(S)=\frac12\lVert S^\top k_j-v_j\rVert_2^2$$

把元素展开：$\ell_j=\frac12\sum_r(\sum_a S_{ar}k_{j,a}-v_{j,r})^2$，因此：

$$\frac{\partial\ell_j}{\partial S_{ar}}=k_{j,a}\left(\sum_b S_{br}k_{j,b}-v_{j,r}\right)$$

$$\nabla_S\ell_j(S)=k_j(S^\top k_j-v_j)^\top$$

在 $\bar S_j$ 处作步长 $\beta_j$ 的一次更新：

$$S_j=\bar S_j-\beta_jk_j(\bar S_j^\top k_j-v_j)^\top=\bar S_j+\beta_j k_j(v_j-\bar S_j^\top k_j)^\top$$

重新合并得到与论文状态朝向一致的形式：

$$S_j=(I-\beta_j k_jk_j^\top)\operatorname{Diag}(\alpha_j)S_{j-1}+\beta_j k_jv_j^\top$$

每个位置读出 $o_j=S_j^\top q_j$。完整算子还包含输入投影、短卷积、归一化、输出门与输出投影；本课实现停在可独立验证的 $o_j$，不把这个核命名为完整 Chimera Block。

### **3. 手算：衰减与纠错不是一回事**

> 取 $S_{j-1}=[2,4]^\top$，$d_k=2,d_v=1$，$\alpha=[0.5,0.25]$，$k=[1,0]^\top$，$v=[3]$，$\beta=0.5$，$q=[1,1]^\top$。
>
> 衰减后 $\bar S=[1,1]^\top$；旧预测 $\bar S^\top k=1$；误差 $v-\hat v=2$。
>
> 写入量 $0.5[1,0]^\top\times2=[1,0]^\top$；新状态 $S_j=[2,1]^\top$；读出 $S_j^\top q=3$。
>
> 只有第一行沿当前 key 修正；第二行保留衰减后的记忆。这不是“把所有 value 通道乘同一个门”，也不是“总是直接覆盖旧值”。

这里恰好使用坐标轴 key；一般 key 会同时更新多行。$\operatorname{Diag}(\alpha)$ 与 $kk^\top$ 通常不交换，因此不能把衰减挪到整个更新的末尾。

### **4. 归一化为什么影响稳定性**

令 $\lVert k\rVert_2=1$。矩阵 $I-\beta kk^\top$ 在 $k$ 方向的特征值为 $1-\beta$，在正交方向为 1。当 $0\leq\beta\leq1$ 时，这个纠错算子不会放大旧状态的欧氏范数；若所有衰减也在 $[0,1]$，两者乘积对旧状态同样非扩张。

这是对 **旧状态传播项** 的有限结论：新写入项仍可累积；网络其余投影和梯度仍可能不稳定。若 key 未归一化，该方向的特征值变为 $1-\beta\lVert k\rVert_2^2$，可以小于 −1，所以“门在 0 到 1”不足以保证稳定。

原始 KDA 的门值是神经网络计算出的，优化内核还可能接收 log-decay 并内部指数化。本课直接传入已构造的 $\alpha,\beta$；迁移到 [官方引出的 FLA KDA 内核](https://github.com/fla-org/flash-linear-attention/tree/main/fla/ops/kda) 时必须重新核对参数语义，不能把 $\alpha$ 当作 log-decay 传进去。

### **5. 线性复杂度、有限容量和并行训练**

每个 token 的 decay、矩阵读取和外积写入约为 $O(d_kd_v)$；固定头数和维度时，全序列为 $O(n d_kd_v)$，状态为 $O(d_kd_v)$。这就是“对长度线性”，并不是没有矩阵乘法，也不是自动优于 GPU 上高效的全注意力。

记忆矩阵没有保存每个历史 token 的独立槽位。很多相似 key 的信息会被投影进同一有限空间；低成本来自压缩，也带来检索干扰。因此 KDA 与全局查询可以互补，而不是一个无损替代另一个。

朴素 Python 循环是理解递推的工具，不适合代表生产性能。递推可写成仿射变换 $S_j=B_jS_{j-1}+U_j$，其中 $B_j=(I-\beta_jk_jk_j^\top)A_j$。两个区间的复合满足：

$$(B_2,U_2)\circ(B_1,U_1)=(B_2B_1,B_2U_1+U_2)$$

复合具有结合性，为扫描／分块计算提供入口；但若显式构造大矩阵乘积，仍会浪费计算。高性能 KDA 使用其结构做分块算法与内核优化，本课没有复刻这些内核，也不以循环耗时评价论文。

## **四、把 KDA 写对｜参考、独立核对和状态边界**

### **1. 参考代码的适用范围**

以下代码单头、批量化，`d_k` 是 key 维，`d_v` 是 value 维，`n` 是长度；多头可增加独立 head 轴或折进批量轴。输入已是预处理后的 query/key/value。本课使用 CPU FP64 做精度检查，测试固定 generator；不下载数据，不读 ArtBench。

函数接受初态便于检查分段扫描。真实独立样本必须重置状态；只有同一样本的连续片段可以继承。对序列显式拒绝空长度，其他形状由接口约定及 PyTorch 运算约束；不承担生产数据加载器的全部验证职责。

In [2]:
def kda_step(
    state: Float[Tensor, "b d_k d_v"],  # noqa: F722
    key: Float[Tensor, "b d_k"],  # noqa: F722
    value: Float[Tensor, "b d_v"],  # noqa: F722
    decay: Float[Tensor, "b d_k"],  # noqa: F722
    beta: Float[Tensor, " b"],  # noqa: F722
) -> Float[Tensor, "b d_k d_v"]:  # noqa: F722
    """先衰减各 key 通道，再用 delta rule 修正关联记忆。

    Args:
        state: 上一个位置的状态，形状为 (b, d_k, d_v)。
        key: 当前 key，形状为 (b, d_k)；本课测试使用单位范数。
        value: 当前要写入的 value，形状为 (b, d_v)。
        decay: 各 key 通道的遗忘系数，形状为 (b, d_k)。
        beta: 每个样本的更新强度，形状为 (b,)。

    Results:
        Tensor: 更新后的 (b, d_k, d_v) 状态，保留自动微分图。
    """
    decayed = decay[..., None] * state
    prediction = torch.einsum("bi,bij->bj", key, decayed)
    correction = value - prediction
    return decayed + beta[:, None, None] * key[..., None] * correction[:, None, :]


def kda_sequence(
    query: Float[Tensor, "b n d_k"],  # noqa: F722
    key: Float[Tensor, "b n d_k"],  # noqa: F722
    value: Float[Tensor, "b n d_v"],  # noqa: F722
    decay: Float[Tensor, "b n d_k"],  # noqa: F722
    beta: Float[Tensor, "b n"],  # noqa: F722
    initial_state: Float[Tensor, "b d_k d_v"] | None = None,  # noqa: F722
) -> tuple[Float[Tensor, "b n d_v"], Float[Tensor, "b d_k d_v"]]:  # noqa: F722
    """按固定扫描顺序更新状态，返回每个位置的未门控读出。

    Args:
        query: 用于读取状态的 query，形状为 (b, n, d_k)，要求 n 大于零。
        key: 当前写入位置的 key，形状为 (b, n, d_k)。
        value: 要记忆的 value，形状为 (b, n, d_v)。
        decay: 通道遗忘系数，形状为 (b, n, d_k)。
        beta: 标量更新强度，形状为 (b, n)。
        initial_state: 分块续接同一样本时的初态；新样本默认从零开始。

    Raises:
        ValueError: 序列为空。

    Results:
        tuple[Tensor, Tensor]:
            output: 每个位置的状态读出，形状为 (b, n, d_v)。
            state: 最终状态，形状为 (b, d_k, d_v)。
    """
    if query.shape[1] == 0:
        raise ValueError("序列长度 n 必须大于零")
    state = initial_state
    if state is None:
        state = query.new_zeros(query.shape[0], key.shape[-1], value.shape[-1])
    outputs = []
    for index in range(query.shape[1]):
        state = kda_step(state, key[:, index], value[:, index], decay[:, index], beta[:, index])
        outputs.append(torch.einsum("bi,bij->bj", query[:, index], state))
    return torch.stack(outputs, dim=1), state


def kda_matrix_reference(
    query: Float[Tensor, "b n d_k"],  # noqa: F722
    key: Float[Tensor, "b n d_k"],  # noqa: F722
    value: Float[Tensor, "b n d_v"],  # noqa: F722
    decay: Float[Tensor, "b n d_k"],  # noqa: F722
    beta: Float[Tensor, "b n"],  # noqa: F722
) -> tuple[Float[Tensor, "b n d_v"], Float[Tensor, "b d_k d_v"]]:  # noqa: F722
    """直接构造矩阵乘积，独立核对 Eq14 的衰减及擦除顺序。

    Args:
        query: (b, n, d_k) query；n 大于零。
        key: (b, n, d_k) key。
        value: (b, n, d_v) value。
        decay: (b, n, d_k) 通道遗忘系数。
        beta: (b, n) 更新强度。

    Results:
        tuple[Tensor, Tensor]: 所有位置的读出与最终状态。
    """
    state = query.new_zeros(query.shape[0], key.shape[-1], value.shape[-1])
    identity = torch.eye(key.shape[-1], dtype=query.dtype, device=query.device)
    outputs = []
    for index in range(query.shape[1]):
        current_key = key[:, index]
        strength = beta[:, index, None, None]
        erase = identity - strength * current_key[..., None] * current_key[:, None, :]
        write = strength * current_key[..., None] * value[:, index, None, :]
        state = erase @ torch.diag_embed(decay[:, index]) @ state + write
        outputs.append((query[:, index, None, :] @ state).squeeze(1))
    return torch.stack(outputs, dim=1), state

### **2. 为什么这组测试能发现方向错误**

逐步实现采用“先预测、再残差写入”；独立对照直接使用矩阵式 $(I-\beta kk^\top)A$。两份代码的运算结构不同，能捕捉常见的转置、广播轴和衰减位置错误。手算案例另外固定一个外部答案，避免两份代码共享错误。

前缀不变测试只证明 **给定 Q/K/V 等输入后的递推** 不读取后缀。若上游空间卷积已把后缀 token 的信息混入 Q/K/V，它不证明对原始 token 同样严格因果。分段一致说明状态传递正确，不说明跨扩散步可以复用。

In [3]:
def check_kda() -> dict[str, float | bool]:
    """检查手算、独立矩阵式、因果前缀、分块、重置与自动微分。

    Raises:
        AssertionError: 任一数值、状态边界或梯度检查不满足预期。

    Results:
        dict[str, float | bool]: 独立数值误差和各项机制检查结果。
    """
    generator = torch.Generator().manual_seed(8)
    dtype = torch.float64
    query = torch.randn(2, 4, 2, generator=generator, dtype=dtype, requires_grad=True)
    key = torch.randn(2, 4, 2, generator=generator, dtype=dtype)
    key = (key / key.norm(dim=-1, keepdim=True)).requires_grad_()
    value = torch.randn(2, 4, 2, generator=generator, dtype=dtype, requires_grad=True)
    decay = (0.2 + 0.6 * torch.rand(2, 4, 2, generator=generator, dtype=dtype)).requires_grad_()
    beta = (0.2 + 0.6 * torch.rand(2, 4, generator=generator, dtype=dtype)).requires_grad_()
    inputs = (query, key, value, decay, beta)
    actual, final_state = kda_sequence(*inputs)
    expected, expected_state = kda_matrix_reference(*inputs)
    torch.testing.assert_close(actual, expected, rtol=1e-12, atol=1e-12)
    torch.testing.assert_close(final_state, expected_state, rtol=1e-12, atol=1e-12)
    # gradcheck 接收 Tensor 元组；只取 sequence 的第一个返回值进行有限差分。
    gradcheck_passed = torch.autograd.gradcheck(lambda *values: kda_sequence(*values)[0], inputs)
    assert gradcheck_passed
    changed = [item.detach().clone() for item in inputs]
    for item in changed:
        item[:, 2:] += 0.1
    assert torch.equal(actual[:, :2].detach(), kda_sequence(*changed)[0][:, :2])
    left, left_state = kda_sequence(*(item[:, :2] for item in inputs))
    right, _ = kda_sequence(*(item[:, 2:] for item in inputs), initial_state=left_state)
    torch.testing.assert_close(torch.cat((left, right), dim=1), actual)
    new_sample, _ = kda_sequence(*(item[:, 2:] for item in inputs))
    assert not torch.allclose(new_sample, actual[:, 2:])
    # 每个 batch 样本独立；分开计算与批量计算应一致。
    for index in range(2):
        isolated, _ = kda_sequence(*(item[index : index + 1] for item in inputs))
        torch.testing.assert_close(isolated, actual[index : index + 1])
    previous = torch.tensor([[[2.0], [-1.0]]], dtype=dtype)
    hand_key = torch.tensor([[0.6, 0.8]], dtype=dtype)
    hand_decay = torch.tensor([[0.5, 0.25]], dtype=dtype)
    hand_value = torch.tensor([[2.0]], dtype=dtype)
    hand_beta = torch.tensor([0.5], dtype=dtype)
    hand = kda_step(previous, hand_key, hand_value, hand_decay, hand_beta)
    torch.testing.assert_close(hand, torch.tensor([[[1.48], [0.39]]], dtype=dtype))
    readout = (hand_key[:, None, :] @ hand).item()
    assert abs(readout - 1.2) < 1e-12
    # beta=0 只衰减；decay=1 且 beta=0 保留原状态，方便定位更新错误。
    torch.testing.assert_close(
        kda_step(previous, hand_key, hand_value, hand_decay, torch.zeros_like(hand_beta)),
        previous * hand_decay[..., None],
    )
    torch.testing.assert_close(
        kda_step(
            previous, hand_key, hand_value, torch.ones_like(hand_decay), torch.zeros_like(hand_beta)
        ),
        previous,
    )
    coordinate = kda_step(
        torch.tensor([[[2.0], [4.0]]], dtype=dtype),
        torch.tensor([[1.0, 0.0]], dtype=dtype),
        torch.tensor([[3.0]], dtype=dtype),
        hand_decay,
        hand_beta,
    )
    torch.testing.assert_close(coordinate, torch.tensor([[[2.0], [1.0]]], dtype=dtype))
    try:
        kda_sequence(*(item[:, :0] for item in inputs))
    except ValueError:
        pass
    else:
        raise AssertionError("空序列应明确拒绝")
    return {
        "matrix_max_error": (actual - expected).abs().max().item(),
        "gradcheck": gradcheck_passed,
        "prefix_causality": True,
        "chunk_carry": True,
        "sample_reset": True,
        "oblique_hand_readout": readout,
        "coordinate_hand_readout": coordinate.sum().item(),
        "empty_sequence_rejected": True,
    }


kda_results = check_kda()
print(kda_results)

{'matrix_max_error': 5.551115123125783e-17, 'gradcheck': True, 'prefix_causality': True, 'chunk_carry': True, 'sample_reset': True, 'oblique_hand_readout': 1.2000000000000002, 'coordinate_hand_readout': 3.0, 'empty_sequence_rejected': True}


实际输出中的误差应接近双精度舍入范围。`gradcheck` 将自动微分与有限差分对照；它只能检查这里给定的微小输入附近。这里没有模型训练损失，也没有生成质量结果。

## **五、MLA｜保留直接检索，同时压缩 key/value 表示**

### **1. 为何还需要少量全局层**

状态记忆的读出依赖此前已压缩进矩阵的信息；全注意力则能为当前 query 单独给每个 token 打分。引入少量全局层，可以让相隔很远的图文位置重新发生直接交互。**Multi-head Latent Attention，MLA** 的重点是压缩 KV 表示，不是把 softmax 取消，也不是把全局层变成线性长度复杂度。[DeepSeek-V2 §2.1](https://arxiv.org/html/2405.04434v5#S2.SS1) 是这一机制的基础阅读。

为避免行／列混乱，下面整节采用 **token 为行向量** 的记法。每头的完整 key 由两部分拼接：从压缩表示升维的 key，与直接从隐藏状态投影、跨头共享的 direct key。Chimera 的 direct key 不使用 RoPE；这不能简化为“没有 direct 分支”。

### **2. 先把压缩路径与非线性说清楚**

设输入 $X\in\mathbb R^{n\times d}$，压缩宽度为 $r$，先求：

$$U=XW_D,\qquad C=\operatorname{RMSNorm}(U)\in\mathbb R^{n\times r}$$

对某个头，取 $W_K\in\mathbb R^{r\times d_\ell}$、$W_V\in\mathbb R^{r\times d_v}$，以及 direct key $K^d\in\mathbb R^{n\times d_d}$：

$$K^\ell=CW_K,\qquad V=CW_V,\qquad K=[K^d,K^\ell],\qquad Q=[Q^d,Q^\ell]$$

于是全局 logits 为：

$$A=\frac{Q^d(K^d)^\top+Q^\ell(CW_K)^\top}{\sqrt{d_d+d_\ell}}+M$$

$M$ 是加性 mask：合法位置为 0，非法位置为 $-\infty$。同一样本内双向，打包样本之间隔离；不应误用语言模型的统一下三角掩码。每行至少保留一个有效 key。随后 $P=\operatorname{softmax}(A)$，输出 $O=P(CW_V)$。

注意，$C$ 已经包含 RMSNorm。它依输入范数而变，不能把 RMSNorm 当作固定矩阵并进 $W_D$。本节只对它之后的线性升维重排乘法。

### **3. 权重吸收的推导与限制**

由矩阵乘法结合律：

$$Q^\ell(CW_K)^\top=(Q^\ell W_K^\top)C^\top$$

$$P(CW_V)=(PC)W_V$$

因此可以先把 query 投影进压缩空间、在那里与 $C$ 打分，再对 $C$ 作加权汇总，最后升维输出。direct 分支的分数仍必须加上，缩放仍为原来的 $1/\sqrt{d_d+d_\ell}$，**不能随吸收后的点积维度改成 $1/\sqrt r$**。

这是同一组参数的代数等价，不表示任意 MHA 权重都可无损压缩成任意低 rank 的 MLA。低 rank 是架构约束；两条路径只是同一个低 rank 架构的计算方式。权重位置若存在额外非线性、RoPE 或依位置变化的变换，不能直接套此结合律。

本课省略线性层 bias、输出投影和多头拼接；这不是声明官方模型没有这些项。有 bias 时必须把各偏置项逐项展开，而不是照抄这份无 bias 公式。现代内核也未必在所有场景都选择吸收路径，是否更快由维度和硬件决定。

### **4. 数值、mask 和梯度一起核对**

代码中的 `d_r` 对应公式 $d_d$（direct 分支宽度），`d_l` 对应 $d_\ell$，`r` 才是 KV 压缩宽度，`n_h` 是头数。参考函数显式保留 RMSNorm 与 direct key；两条路径共用参数但分别计算。测试使用一个 block-diagonal mask 模拟两段独立样本，既核对输出，也核对输入和升维矩阵的梯度。额外扰动隔离样本的内容，检查另一段输出不变。接口只接受由0／负无穷构成的浮点加性 mask；bool、NaN、正无穷、错误尺寸、全遮挡行及非零有限偏置均明确拒绝。这里刻意不支持一般 attention bias，以免把另一种 mask 约定误传成0/1分数。

In [4]:
def rms_unit(
    latent: Float[Tensor, "b n r"],  # noqa: F722
    epsilon: float = 1e-6,
) -> Float[Tensor, "b n r"]:  # noqa: F722
    """计算单位 gain 的 RMSNorm，保留非线性归一化的位置。

    Args:
        latent: 下投影后的原始 KV 潜变量，形状为 (b, n, r)。
        epsilon: 防止除零的正数。

    Results:
        Tensor: 归一化后用于升维或缓存的 C，形状为 (b, n, r)。
    """
    return latent * torch.rsqrt(latent.square().mean(dim=-1, keepdim=True) + epsilon)


def mla_expanded(
    query_direct: Float[Tensor, "b n_h n d_r"],  # noqa: F722
    query_latent: Float[Tensor, "b n_h n d_l"],  # noqa: F722
    direct_key: Float[Tensor, "b n d_r"],  # noqa: F722
    raw_latent: Float[Tensor, "b n r"],  # noqa: F722
    up_key: Float[Tensor, "n_h r d_l"],  # noqa: F722
    up_value: Float[Tensor, "n_h r d_v"],  # noqa: F722
    mask: Float[Tensor, "n n"] | None = None,  # noqa: F722
) -> Float[Tensor, "b n_h n d_v"]:  # noqa: F722
    """显式还原每个头的 latent key 和 value，作为 MLA 代数对照。

    Args:
        query_direct: 对应 shared direct key 的 query 部分，形状为 (b, n_h, n, d_r)。
        query_latent: 对应 latent key 的 query 部分，形状为 (b, n_h, n, d_l)。
        direct_key: 各头共享的 direct key，形状为 (b, n, d_r)。
        raw_latent: 未归一化 KV 潜变量，形状为 (b, n, r)。
        up_key: 每个头的 key 升维矩阵，形状为 (n_h, r, d_l)。
        up_value: 每个头的 value 升维矩阵，形状为 (n_h, r, d_v)。
        mask: 可选加性 mask，形状 (n, n)，合法位置为 0，禁止位置为 -inf。

    Raises:
        ValueError: mask 形状／浮点类型错误、含0/-inf以外值，或任一行全部被遮挡。

    Results:
        Tensor: 输出投影前的各头注意力结果，形状为 (b, n_h, n, d_v)。
    """
    latent = rms_unit(raw_latent)
    key = torch.einsum("bnr,hrd->bhnd", latent, up_key)
    value = torch.einsum("bnr,hrd->bhnd", latent, up_value)
    scale = math.sqrt(query_direct.shape[-1] + query_latent.shape[-1])
    scores = (
        query_direct @ direct_key[:, None].transpose(-1, -2) + query_latent @ key.transpose(-1, -2)
    ) / scale
    if mask is not None:
        if (
            mask.shape != scores.shape[-2:]
            or not mask.is_floating_point()
            or not ((mask == 0) | torch.isneginf(mask)).all()
            or not (mask == 0).any(dim=-1).all()
        ):
            raise ValueError("mask 须为浮点 n×n，仅含0/-inf，每行至少一个0")
        scores = scores + mask
    return scores.softmax(dim=-1) @ value


def mla_absorbed(
    query_direct: Float[Tensor, "b n_h n d_r"],  # noqa: F722
    query_latent: Float[Tensor, "b n_h n d_l"],  # noqa: F722
    direct_key: Float[Tensor, "b n d_r"],  # noqa: F722
    raw_latent: Float[Tensor, "b n r"],  # noqa: F722
    up_key: Float[Tensor, "n_h r d_l"],  # noqa: F722
    up_value: Float[Tensor, "n_h r d_v"],  # noqa: F722
    mask: Float[Tensor, "n n"] | None = None,  # noqa: F722
) -> Float[Tensor, "b n_h n d_v"]:  # noqa: F722
    """吸收线性升维矩阵，直接在归一化 KV 潜空间计算。

    Args:
        query_direct: (b, n_h, n, d_r) 的 direct-query 分量。
        query_latent: (b, n_h, n, d_l) 的 latent-query 分量。
        direct_key: (b, n, d_r) 的跨头共享 direct key。
        raw_latent: (b, n, r) 的原始 KV 潜变量；仍须先 RMSNorm。
        up_key: (n_h, r, d_l) 的线性 key 升维矩阵。
        up_value: (n_h, r, d_v) 的线性 value 升维矩阵。
        mask: 可选加性 mask，形状 (n, n)，每行至少一个合法 key。

    Raises:
        ValueError: mask 形状／浮点类型错误、含0/-inf以外值，或任一行全部被遮挡。

    Results:
        Tensor: 与显式实现等价的各头输出，形状为 (b, n_h, n, d_v)。
    """
    latent = rms_unit(raw_latent)
    # 只把线性的 key 升维吸收到 query 侧，不跨过 RMSNorm。
    compressed_query = torch.einsum("bhnd,hrd->bhnr", query_latent, up_key)
    scale = math.sqrt(query_direct.shape[-1] + query_latent.shape[-1])
    scores = (
        query_direct @ direct_key[:, None].transpose(-1, -2)
        + compressed_query @ latent[:, None].transpose(-1, -2)
    ) / scale
    if mask is not None:
        if (
            mask.shape != scores.shape[-2:]
            or not mask.is_floating_point()
            or not ((mask == 0) | torch.isneginf(mask)).all()
            or not (mask == 0).any(dim=-1).all()
        ):
            raise ValueError("mask 须为浮点 n×n，仅含0/-inf，每行至少一个0")
        scores = scores + mask
    latent_context = scores.softmax(dim=-1) @ latent[:, None]
    return torch.einsum("bhnr,hrd->bhnd", latent_context, up_value)

In [5]:
def check_mla() -> dict[str, float | bool]:
    """独立比较显式升维与权重吸收路径的输出及所有输入梯度。

    Raises:
        AssertionError: FP64 输出或梯度误差超过预设容差。

    Results:
        dict[str, float | bool]: 最大输出误差、最大梯度误差与双向注意力检查结果。
    """
    generator = torch.Generator().manual_seed(19)
    shapes = [(2, 2, 4, 2), (2, 2, 4, 3), (2, 4, 2), (2, 4, 2), (2, 2, 3), (2, 2, 3)]
    inputs = tuple(
        torch.randn(*shape, generator=generator, dtype=torch.float64, requires_grad=True)
        for shape in shapes
    )
    mask = torch.full((4, 4), -torch.inf, dtype=torch.float64)
    mask[:2, :2] = 0.0
    mask[2:, 2:] = 0.0
    expanded = mla_expanded(*inputs, mask=mask)
    absorbed = mla_absorbed(*inputs, mask=mask)
    torch.testing.assert_close(expanded, absorbed, rtol=1e-12, atol=1e-12)
    probe = torch.randn(expanded.shape, generator=generator, dtype=torch.float64)
    gradients_a = torch.autograd.grad((expanded * probe).sum(), inputs)
    gradients_b = torch.autograd.grad((absorbed * probe).sum(), inputs)
    maximum = 0.0
    for gradient_a, gradient_b in zip(gradients_a, gradients_b, strict=True):
        torch.testing.assert_close(gradient_a, gradient_b, rtol=1e-10, atol=1e-10)
        maximum = max(maximum, (gradient_a - gradient_b).abs().max().item())
    # 双向 attention 可使位置 0 受到最后一个 key/value 的影响。
    changed = [item.detach().clone() for item in inputs]
    changed[3][:, -1] += torch.tensor([0.3, -0.2], dtype=torch.float64)
    # 带 packed mask 时，另一样本的变化不应污染前两项。
    torch.testing.assert_close(mla_expanded(*changed, mask=mask)[:, :, :2], expanded[:, :, :2])
    unrestricted = mla_expanded(*inputs)
    torch.testing.assert_close(unrestricted, mla_absorbed(*inputs), rtol=1e-12, atol=1e-12)
    assert not torch.allclose(mla_expanded(*changed)[:, :, 0], unrestricted[:, :, 0])
    nan_mask = mask.clone()
    nan_mask[0, 1] = torch.nan
    positive_inf_mask = mask.clone()
    positive_inf_mask[0, 1] = torch.inf
    invalid_masks = [
        torch.ones(4, 4, dtype=torch.bool),
        nan_mask,
        positive_inf_mask,
        torch.zeros(3, 3, dtype=torch.float64),
        torch.full_like(mask, -torch.inf),
        torch.ones_like(mask),
    ]
    for invalid in invalid_masks:
        for implementation in (mla_expanded, mla_absorbed):
            try:
                implementation(*inputs, mask=invalid)
            except ValueError:
                pass
            else:
                raise AssertionError("不符合加性 mask 契约的输入必须被拒绝")
    return {
        "output_max_error": (expanded - absorbed).abs().max().item(),
        "gradient_max_error": maximum,
        "bidirectional": True,
        "packed_isolation": True,
        "invalid_masks_rejected": len(invalid_masks) * 2,
    }


mla_results = check_mla()
print(mla_results)

{'output_max_error': 1.3322676295501878e-15, 'gradient_max_error': 2.4424906541753444e-15, 'bidirectional': True, 'packed_isolation': True, 'invalid_masks_rejected': 12}


输出等价没有展示真实 cache 系统。若实际实现先展开完整 K/V 再存储，便没有获得声称的压缩缓存收益。本课用显式 $n\times n$ softmax 便于检查，不能把这里的显存用量当作融合内核的显存。

### **5. 混合以后为什么整体仍不是线性复杂度**

固定层数中若四层有三层 KDA、一层全局 MLA，可以粗略写为：

$$C(n)\approx\frac34 a n+\frac14 b n^2+c n d^2$$

第一项代表固定维度的递推工作，第二项代表仍存在的全局交互，第三项归纳投影、FFN 等计算。只要全局层比例非零且不随长度消失，渐近复杂度仍包含 $O(n^2)$；主要改变的是系数和部分状态存储。

FlashAttention 通过分块与在线 softmax 减少显式 score 存储，不改变精确全注意力的二次交互算术。实现层面必须分别问：**省了哪些常驻张量，省了哪些运算，减少了哪些内存读写？** 一个“O(n)”标签不能回答三件事。

## **六、NoPE 与短卷积｜没有显式位置编码，为什么仍知道顺序**

### **1. 从排列等变性开始理解**

若只有无位置、无顺序 mask 的 self-attention，把输入 token 同时按置换矩阵 $\Pi$ 重排，输出也会相同重排。因为 $Q'=\Pi Q,K'=\Pi K,V'=\Pi V$，所以：

$$\operatorname{softmax}(\Pi QK^\top\Pi^\top)\Pi V=\Pi\operatorname{softmax}(QK^\top)V$$

它能分辨“内容不同”，但没有单独的坐标依据来认定哪个 token 是左上角。Chimera 的 **NoPE** 只表示不注入显式位置 embedding／RoPE；递推顺序、边界和卷积布局仍然携带位置信息。所以把旧 B1 的 RoPE 一删，并不等于复现这个方案。

### **2. 扫描顺序与空间邻域不是同一个东西**

视觉网格按时间优先、行优先展平，可用零起始映射 $j=f h w+yw+x$。相邻 token 可能来自一行末尾和下一行开头；它们在线性序列上相邻，但在二维图上未必邻近。普通 1D short convolution 无法自动恢复这种空间几何。

modality-aware short convolution 的思路是在原生布局中做局部混合，再回到序列：文本沿文本轴做因果 1D depthwise 卷积；视觉回到时间、高、宽网格做 depthwise 3D 卷积。depthwise 表示每个通道有自己的局部核，不在这一步跨通道求和；通道交互由投影等其他层承担。

图像可视为只有一帧的视频。处理 packed sequence 时，必须知道每个样本、每段模态的位置；卷积核不能跨越文本／图像边界，也不能跨独立样本边界。[Chimera §3.4](https://arxiv.org/html/2607.28611v1#S3.SS4) 给出了该布局约束；本课不声称已运行其融合内核。

### **3. “因果”的三个层次要分别说**

| 层次 | 能保证什么 | 不能推出什么 |
| --- | --- | --- |
| KDA 递推核 | 当前 Q/K/V 输入只与此前状态组合 | 原始图像信息严格只从 raster 前缀流入 |
| 视觉短卷积 | 时间轴不看未来帧 | 同一帧的右侧／下方邻居不能参与 |
| 完整去噪器 | 所有位置的噪声状态共同决定速度 | 网络是逐 token 自回归生成器 |

视觉卷积的时间 causal 与二维空间邻域并存；周期性 MLA 还允许双向全局信息。因此完整主干不能因为有 causal KDA 就复用一个语言模型的 AR mask 或声称严格在线生成。

对独立样本，需要同时隔离全局 attention mask、KDA 状态、卷积边界。只隔离其中一个仍会泄漏。对 padding，需要防止无效位置改变状态或参与聚合；“输出时乘 mask”不能消除已经发生的污染。

### **4. 这里尤其不能照搬 S5-02 的条件 cache**

S5-02 讨论的缓存成立需要条件分支在计算图上独立于当前图像噪声及时间调制等变化量。Chimera 的全局双向 MLA 可让文本隐藏状态读入视觉信息；共享时间调制还可能直接随 $\tau$ 变化。即使初始 T5/CLIP 特征不变，也不能推出每层文本 K/V 不变。

可以缓存冻结文本塔的输出；不能未经依赖审计缓存整个主干内的文本状态。更不能在两次去噪调用之间保留已经吸收视觉噪声的 KDA 状态。CFG 有条件与空条件分支也不能共享这些状态。若未来要支持流式视频，须单独设计时间依赖和训练协议，并重新验证，而不是开一个 cache 开关。

## **七、从算子到深层网络｜调制、残差与容量**

### **1. 一个块究竟怎样连起来**

从结构看，可将子层阅读成：从残差状态读表示 → pre-norm → 时间调制 → attention → post-norm → 时间门控 → 写回残差；随后 FFN 也经过归一化、门控与写回。这里 **sandwich normalization** 指算子前后都做归一化，并不等于“对输入重复归一化两遍”。

RMSNorm 不减均值；对向量 $x\in\mathbb R^d$，基本形式是：

$$\operatorname{RMSNorm}(x)=g\odot\frac{x}{\sqrt{\frac1d\sum_{a=1}^d x_a^2+\varepsilon}}$$

$g$ 为可学习 gain。将其称为 LayerNorm 并套上“零均值”的解释是不准确的。时间调制可以写成 $\tilde x=\operatorname{RMSNorm}(x)\odot(1+\gamma(\tau))+\delta(\tau)$，另用 $g_A(\tau),g_F(\tau)$ 门控更新；同一个字母不能同时代表归一化 gain 与时间门，代码应分名。

论文块的 attention 输入有 scale/shift，两条残差各有 gate；这与原始 DiT 每个子层均有 scale/shift/gate 的六路 adaLN-Zero 不完全相同。**不要仅因名字相似，替它补上未核实的 FFN 调制或零初始化。**

### **2. iHC 的“多流”是什么**

先看普通残差 $x'=x+\Delta$。若保留 $m$ 份表示 $R_1,\ldots,R_m$，可定义：

$$\tilde x=\sum_{a=1}^m h_{\mathrm{pre},a}(R)R_a,\qquad R'_a=R_a+h_{\mathrm{post},a}(R)\Delta$$

读系数决定子层看哪种组合，写系数决定把同一个更新分配给哪些副本。“identity”指旧状态沿恒等路径保留；不代表 read/write 系数永远固定，也不保证整个 block 恒等。

若初态都是同一个 $x$，读系数为 $1/m$、写系数为 1，则 $\tilde x=x$，每个副本更新成 $x+\Delta$，退化为熟悉的残差路径。这一手算解释了为什么初始化时接近普通残差是合理设计。真实可学习系数不是一直相等；采用 sigmoid 的独立系数也不自动保证读系数和为 1。

这与 S6 的文本／图像双流不同：iHC 在 **每一个 token 内部** 复制残差表示，模态 token 仍处于共同序列。它会增加激活存储；固定小 $m$ 的额外运算不等于零开销。本课只讲机制，不增加项目 iHC 实现。

### **3. MoE：激活参数少，不代表模型装得下**

Mixture of Experts 用 router 为每个 token 选择少数 FFN 专家，输出是所选专家结果的加权和：

$$y=\sum_{i\in\mathcal T(x)}g_i(x)E_i(x)$$

其中 $\mathcal T$ 是 top-k 专家集合；它不是只挑“正确专家”的监督标签。不同 token 可选择不同专家，训练中需要检查负载分布与塌缩。总参数决定存储与潜在容量；激活参数更接近单 token 的专家计算，但并不包含所有通信、dispatch、padding 及内核调度开销。[DeepSeek-V3](https://arxiv.org/abs/2412.19437) 是偏置式负载平衡的背景资料；不要将其完整配方原样归给 Chimera。

若一个 dense FFN 的中间宽度为 $f$，把每个专家宽度设为 $f/k$、每次激活 $k$ 个，则所选专家的主体乘加量可以近似匹配 dense FFN；若共有 $e$ 个专家，存储却约按 $e/k$ 增长。示例 $e=8,k=2$ 时，专家主体参数约为 dense 的 4 倍，激活计算约为 dense 的 1 倍；路由等开销另计。

所有专家可能仍须常驻 GPU，优化器也需要管理可训练参数；“2B active”不能据此按2B参数估算整机显存。单卡 MiniGen 应先用 dense FFN 研究一个算子差异，避免同时把容量、路由与负载平衡引入对照。

对应到Chimera，原版首末层保留dense SwiGLU，中间层用56个专家、每token选择8个，不设shared expert；最终配置中单专家中间宽度为8192/8=1024。选择和加权要分开：router给出sigmoid分数 $r_i$，平衡偏置 $b_i$ 只进入top-k选择；实际组合权重来自未加偏置的分数：

$$\mathcal T=\operatorname{TopK}(r+b,8),\qquad g_i=\sqrt8\frac{r_i}{\sum_{j\in\mathcal T}r_j}\quad(i\in\mathcal T)$$

因此权重和是 $\sqrt8$，不是1；不能把它当作标准概率混合。平衡偏置按负载调整，不通过此偏置新增loss项。若复现时把偏置也加进组合权重，便改变了算子；若只说“参考DeepSeek”而不核对这些细节，也不足以证明配方一致。[来源：§3.5](https://arxiv.org/html/2607.28611v1#S3.SS5)


## **八、HeteroP 与 Scaling｜为什么不能只把模型做大**

### **1. 尺度变化会改变优化，不只改变容量**

设模型加宽后表现更差，有两种解释：架构容量没有帮助，或原学习率和初始化已不适合它。若不区分，就会把优化失败误认为 scaling law。**参数化** 在这里指初始化、前向缩放、优化器设置如何随宽度／深度共同变化，不是换一种文件格式。

HeteroP 的重要思路是给不同参数组按各自有效 fan-in 设置比例，而不是让所有矩阵跟着统一 hidden width 比例变化。输入是 hidden、低秩 latent 或专家中间层时，fan-in 可能完全不同。原论文 §4.1 的表1还区分输入、隐藏、readout、归一化和深度规则；不能从某一个 LR 公式反推全套配置。

### **2. 用一个小例子读懂“按模块缩放”**

令某隐藏矩阵的 fan-in 从128增至256，则 $m_W=2$；层数从4到8，则 $m_L=2$。若示例采用隐藏权重的初始化方差与 LR 都除以 $m_W$，weight decay 乘 $m_W$：

$$\eta'=\eta/2,\qquad\lambda'=2\lambda,\qquad(\sigma')^2=\sigma^2/2$$

因此 $\eta'\lambda'=\eta\lambda$，AdamW 的单步乘性收缩因子 $1-\eta\lambda$ 保持不变；标准差应乘 $1/\sqrt2$，不能误乘 $1/2$。这个结论只解释一项设计动机，不证明完整训练等价。

若另一矩阵输入的低秩维度保持64不变，它的 fan-in 比例就是1；不能因 hidden 变宽就统一除以2。原方案对残差更新还有深度比例，对 AdamW epsilon 有角色相关规则。真正复现应为每个参数命名、分类并验证公式；本课不将这一示例注册成项目优化器规则。

表1中的深度规则将每个attention／FFN残差更新乘 $m_L^{-1}$，再写入iHC；它不要求把该组LR再统一除以深度。残差块张量的AdamW epsilon按 $m_W^{-1}m_L^{-1}$，输入／readout的epsilon按 $m_W^{-1}$。上述 $m_W=m_L=2$ 的隐藏矩阵例子中，残差系数为1/2，epsilon为基础值的1/4；输入／readout不能套上隐藏矩阵的完整规则。epsilon、初始化、前向系数与LR一起定义参数化，只抄其中一项不能叫HeteroP复现。


### **3. 从固定预算推导参数与数据的平衡**

把参数规模记作 $N$，累计处理 token 数记作 $D$，避免与前文序列长度 $n$ 混用。一种常见经验拟合为：

$$L(N,D)=E+A N^{-a}+B D^{-b},\qquad A,B,a,b>0$$

$E$ 是拟合的不可约项，后两项分别描述有限容量与训练数据量的代价。它是需要数据检验的近似模型，不是自然定律。[Chinchilla §3](https://arxiv.org/html/2203.15556v1#S3) 提供了固定预算分配的经典研究框架。

为手算，假设训练计算 $C=\kappa ND$，$\kappa$ 暂为常数。代入 $D=C/(\kappa N)$：

$$L(N)=E+A N^{-a}+B(\kappa N/C)^b$$

对 $N$ 求导并令其为零：

$$-aA N^{-a-1}+bB(\kappa/C)^bN^{b-1}=0$$

$$N_*^{a+b}=\frac{aA}{bB}(C/\kappa)^b$$

$$N_*\propto C^{b/(a+b)},\qquad D_*\propto C^{a/(a+b)}$$

如果 $a=b$，预算翻4倍时最优模型规模与累计 token 量都约翻2倍，而不是模型翻4倍、数据不动。这只是给定拟合和成本假设后的结论；真实数据重复、噪声目标、长序列 attention、MoE dispatch 都会影响假设。不能把语言模型的某个固定 token/parameter 比率机械套到 MiniGen。

### **4. MiniGen 应借鉴的是实验纪律**

30M 的 B0/B1/B2 之间不是同架构尺度扫描；不能用三个不同模型点拟合一个可靠 scaling law。有限预算更应该先保证同目标、同数据身份、相同调参预算，再比较等训练步和等实测GPU时间的曲线。

重复训练同一张图虽然可以得到不同噪声，但不等于新增独立图像。累计视觉 token 次数、唯一训练图像数、epoch 三者都应记录。若未来做10M／30M／50M，同架构家族也需要多个预算点、稳定优化和拟合不确定性；本课不触发这些训练。

## **九、论文证据审查｜不要把不同收益合成一个数字**

### **1. 看到倍率，先找分子、分母和实验对象**

[Chimera v1](https://arxiv.org/html/2607.28611v1#S5) 的架构、优化和数据规模因素同时影响结果。本课记录两处特别容易读错的证据：

| 原文位置与数字 | 正确读法 | 不支持的推断 |
| --- | --- | --- |
| 摘要／引言完整方案7.3×；正文§5.5–5.6共同loss=0.149给出4.29e20与6.27e19 FLOPs | 两数相除约6.84，正文约6.8×；版本内部存在口径差异，保留差异 | 不把7.3×直接写成推理速度，或替作者臆测差异原因 |
| 引言脚注及§5.6在约255k token给出2.14×前向速度 | 特定A100、BF16、batch1、匹配主干和测量范围的结果，且排除cache构建 | 不外推到本机CPU、333 token、训练反向或完整采样 |

前一行可用下面的纯算术复算；没有重新训练或测量论文模型。它提醒我们：论文报告的事实、自己的计算和自己的实测必须分开。

$$\frac{4.29\times10^{20}}{6.27\times10^{19}}\approx6.8421$$

长视频外推结果也要检查时长、分辨率、基线原生设置、最后一段还是全视频的指标。一个帧分布指标不能独自证明时序一致性；去掉显式位置编码也不保证无限外推。

### **2. 官方代码证据的边界**

本轮核查论文链接、[首作主页](https://chongjiange.github.io/) 及[共同作者专题页](https://stevewongv.github.io/research/chimera-arxiv2026/)，未取得可对应 Chimera v1 的完整官方模型／训练源码。因此本课将架构事实定位到论文公式与章节，不编造 `ChimeraBlock` 文件路径，也不声称完成官方权重推理。

[Kimi Linear 作者仓库](https://github.com/MoonshotAI/Kimi-Linear) 链接了 [FLA KDA](https://github.com/fla-org/flash-linear-attention/tree/main/fla/ops/kda)，可用于后续理解算子接口与高性能实现；这是底层机制参考，不是 Chimera 官方仓库。DeepSeek-V2 是 MLA 的基础来源，也不是本课视觉模型的实现。参考代码是依据数学重新写的教学实现，尚未与生产 CUDA kernel 做数值 parity。

### **3. 一个可信研究卡应怎样收尾**

| 问题 | 本课判断 |
| --- | --- |
| 它想解决什么 | 长视觉序列的计算／记忆成本，以及规模变化的优化与预算分配 |
| 最值得理解什么 | 固定状态和逐项检索的互补，NoPE依赖哪些其他结构 |
| 本课已验证什么 | 两个简化算子的代数、梯度和边界；论文数字的有限复算 |
| 对MiniGen最不确定什么 | 短序列下是否有实际速度收益，以及精细属性绑定是否受损 |
| 下一项高价值证据是什么 | 用户另行启动后，在同设备测完整块／训练步，而非只数attention矩阵 |

## **十、回到256px｜如果以后试 Chimera-Lite，怎样避免实验失控**

### **1. 先验证瓶颈，再选择一个因素**

这节是 **未来执行协议**，本次没有运行正式架构比较。先读 S7 的 profiler、预算和审查流程。若主耗时来自数据读取、VAE重复编码或FFN，换attention可能无助于端到端吞吐。用真实n_t分布而非永远77的假设，记录实际文本padding开销。

第一组可测因素是“是否以状态算子替换部分全局层”。保留相同数据、Flow目标、patch、条件编码、precision、优化／EMA协议与dense FFN，约束可训练参数匹配。若替换后参数变化，先说明匹配办法，不用不同参数量结果直接归因于算子。

NoPE是第二个独立因素；不能同时删除RoPE、换KDA、加卷积、加MoE、改学习率，再把收益都归给“混合注意力”。若新算子必须依赖局部卷积才能稳定，可将其明确命名为“算子＋局部结构配方”，再用额外消融拆分，避免伪装单因素。

### **2. 性能测量分四层**

| 层次 | 测什么 | 必须固定／记录 |
| --- | --- | --- |
| 算子 | forward及forward+backward | 同shape/dtype/device，数值精度，实际后端 |
| 完整块 | 投影、norm、FFN、门控和布局开销 | hidden、heads、参数量、是否融合 |
| 训练步 | 多层、loss、optimizer、EMA | 有效batch、累积、checkpointing、data等待 |
| 完整采样 | NFE次去噪、CFG、文本编码与VAE解码 | prompt/noise、solver、NFE、CFG分支、batch |

CUDA先预热再同步计时；记录多次样本的中位数与离散程度，峰值allocated/reserved说明口径，先完成数值一致性再看速度。CPU参考与GPU融合实现不构成公平性能对照。编译时间和首次加载单列，OOM作为结果保留，不能只报告成功长度。

可按333、1k、4k等长度做性能压力测试，但长长度只说明形状下的系统行为；不代表训练过高分辨率，也不代表有长视频能力。只有相同工作量与语义的运行才直接比较；如果语义不同，就明确比较的是两套配方。

### **3. 质量与故障如何逐步诊断**

| 观察 | 首先排查 | 再决定 |
| --- | --- | --- |
| KDA 输出爆炸／NaN | key归一化、门范围、log-decay语义、FP32状态、梯度范数 | 才调整学习率／裁剪；不要靠clamp隐藏错误 |
| 计数或精细绑定退化 | 数据描述质量、相同noise/时间条件、全局层比例、状态干扰 | 增加检索层是否值得成本 |
| 换图／换prompt后串样本 | KDA状态、conv边界、packed mask、CFG分支 | 建立独立样本不变性测试 |
| 理论FLOPs更低但更慢 | 小矩阵利用率、Python循环、kernel launch、gather/scatter | 先看完整训练步的主要耗时 |
| 长序列图像布局差 | 展平顺序、空间邻域、位置机制、分布外分辨率 | 不用“没有RoPE所以能外推”解释失败 |
| 缓存后输出不同 | 图像／时间／分支是否改变中间条件表示 | 关闭非法cache并重做依赖检查 |

仍沿用冻结验证方案、测试隔离、相同采样预算、失败样本和逐seed记录。尚未完成B0/B1/B2正式结果之前，本课研究卡不能替代任何P3–P5验收；也不会改变现有Caption质量问题的处理优先级。

## **十一、只亲手写一个精髓｜一步 KDA 纠错**

### **1. 任务、边界与验收**

你只需完成下面函数的主体：先按key行衰减旧状态，计算当前key的旧预测，再沿残差写入新状态。不要复制整套模型，不需要训练。输入为一个batch的一步状态，输出为同shape的新状态；这项练习刻意比前面的序列扫描更小。

验收必须覆盖手算、非坐标key、零写入门、非均匀衰减、输入未原地改变以及梯度。代码中的测试入口不会把TODO自动换为参考实现；未实现只显示“待完成”，任何真正的数值或形状错误都会向上抛出。

In [6]:
def personal_kda_step(
    state: Float[Tensor, "b d_k d_v"],  # noqa: F722
    key: Float[Tensor, "b d_k"],  # noqa: F722
    value: Float[Tensor, "b d_v"],  # noqa: F722
    decay: Float[Tensor, "b d_k"],  # noqa: F722
    beta: Float[Tensor, " b"],  # noqa: F722
) -> Float[Tensor, "b d_k d_v"]:  # noqa: F722
    """个人练习：自行写出先衰减、再 delta 修正的单步状态更新。

    Args:
        state: (b, d_k, d_v) 旧状态。
        key: (b, d_k) 当前 key。
        value: (b, d_v) 当前 value。
        decay: (b, d_k) 通道遗忘系数。
        beta: (b,) 更新强度。

    Raises:
        NotImplementedError: 个人主体尚未实现；不会调用教学参考替代。
    """
    raise NotImplementedError("待你实现：KDA 单步状态更新")


def check_personal_step() -> None:
    """验收个人一步实现的手算、矩阵式、零门、输入保护与有限差分。

    Raises:
        NotImplementedError: 个人函数未实现，交给外层显示待做。
        AssertionError: 形状、精度、梯度或输入保护不满足约定。
        RuntimeError: 个人实现计算错误或不可微时由 PyTorch 透传。
    """
    dtype = torch.float64
    state = torch.tensor([[[2.0], [-1.0]]], dtype=dtype)
    key = torch.tensor([[0.6, 0.8]], dtype=dtype)
    value = torch.tensor([[2.0]], dtype=dtype)
    decay = torch.tensor([[0.5, 0.25]], dtype=dtype)
    beta = torch.tensor([0.5], dtype=dtype)
    arguments = (state, key, value, decay, beta)
    untouched = tuple(item.clone() for item in arguments)
    actual = personal_kda_step(*arguments)
    assert actual.shape == state.shape
    torch.testing.assert_close(actual, torch.tensor([[[1.48], [0.39]]], dtype=dtype))
    for original, saved in zip(arguments, untouched, strict=True):
        assert torch.equal(original, saved), "个人函数不得原地修改输入"
    generator = torch.Generator().manual_seed(81)
    state = torch.randn(2, 3, 2, generator=generator, dtype=dtype)
    key = torch.randn(2, 3, generator=generator, dtype=dtype)
    key = key / key.norm(dim=-1, keepdim=True)
    value = torch.randn(2, 2, generator=generator, dtype=dtype)
    decay = torch.rand(2, 3, generator=generator, dtype=dtype)
    beta = torch.rand(2, generator=generator, dtype=dtype)
    erase = torch.eye(3, dtype=dtype) - beta[:, None, None] * key[:, :, None] * key[:, None, :]
    expected = erase @ torch.diag_embed(decay) @ state
    expected += beta[:, None, None] * key[:, :, None] * value[:, None, :]
    torch.testing.assert_close(personal_kda_step(state, key, value, decay, beta), expected)
    torch.testing.assert_close(
        personal_kda_step(state, key, value, decay, torch.zeros_like(beta)),
        decay[:, :, None] * state,
    )
    differentiable = tuple(
        item.detach().requires_grad_() for item in (state, key, value, decay, beta)
    )
    assert torch.autograd.gradcheck(personal_kda_step, differentiable)


try:
    check_personal_step()
except NotImplementedError as error:
    print(f"个人练习状态：待做；{error}")
else:
    print("个人一步核心检查通过；理解验收和工程集成仍须分别进行。")

个人练习状态：待做；待你实现：KDA 单步状态更新


### **2. 怎样说明自己真正理解了**

合上代码，解释 $\bar S^\top k$ 为什么是value预测、外积为什么是 `d_k × d_v`、为什么先衰减而不是最后统一乘门。再手算一个不与坐标轴对齐的key，并预测写入会影响哪些行。最后说清：这个练习通过，仅验证个人一步核心；尚未证明多头网络、卷积、Flow训练或生成能力。

## **十二、最终回顾｜把架构选择变成可验证的判断**

### **1. 自测与参考答案**

**问题1：三层KDA加一层MLA，整体是线性复杂度吗？**

答案：不是。固定比例的全局层仍有二次长度项。若长度很短，投影和FFN也可能主导耗时；复杂度类别与实测速度都要分别说明。

**问题2：NoPE为什么仍能分辨顺序？**

答案：递推顺序、卷积的原生网格和边界提供了结构。一个没有这些结构的裸全注意力去掉RoPE，不能获得同样性质；没有显式位置相位也不保证任意长度泛化。

**问题3：MLA吸收后点积宽度变成r，为什么还用旧scale？**

答案：结合律保持了原logits的分子；原模型的温度为 $1/\sqrt{d_d+d_\ell}$。改为 $1/\sqrt r$ 是改变模型，不再是代数等价。

**问题4：既然文本塔输出固定，为什么不能缓存所有文本K/V？**

答案：主干的文本表示可能通过双向MLA依赖图像，并通过调制依赖扩散时间。必须检查中间表示的计算依赖，不能只看最初输入是否相同。

**问题5：模型更大却更差，怎样避免错误结论？**

答案：先排除优化超参、训练token数、数据噪声与预算不匹配。要研究scaling，就在同架构家族和可比优化条件下测多个规模、多个预算点；不能从三个不同架构点作普适外推。

**问题6：你现在最应该做的是实现完整Chimera吗？**

答案：不是当前工程任务。先完成主线的真实训练、条件利用与成本证据。若性能剖析确实显示attention瓶颈，再从一个可测因素开始；如果短序列下没有收益，保留简单模型就是有证据的选择。

### **2. 阅读顺序与来源定位**

| 来源 | 本课对应位置 | 阅读时必须回答 |
| --- | --- | --- |
| [Chimera v1](https://arxiv.org/html/2607.28611v1)，§3、§4、§5 | 系统地图、原方案边界、证据审查 | 哪些是架构事实，哪些是特定实验结果？ |
| [Kimi Linear v1](https://arxiv.org/html/2510.26692v1)，§2–4 | KDA推导、分块与门值 | delta更新与简单累加有何区别？ |
| [Kimi作者仓库](https://github.com/MoonshotAI/Kimi-Linear) → [FLA KDA](https://github.com/fla-org/flash-linear-attention/tree/main/fla/ops/kda) | 后续内核阅读入口 | 接口输入是gate还是log-gate，state轴顺序是什么？ |
| [DeepSeek-V2 v5](https://arxiv.org/html/2405.04434v5)，§2.1及附录C | MLA低秩与吸收 | 哪些线性层能吸收，位置分支怎样保留？ |
| [Chinchilla v1](https://arxiv.org/html/2203.15556v1)，§3 | 固定预算推导 | 什么是假设，什么是拟合结果？ |
| [PyTorch 2.14 SDPA](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html) | mask、scale与后端 | bool mask中True是否允许参与，dropout怎样关闭？ |

SDPA的bool mask为True表示允许参与；本课参考用加性mask避免与某些其他attention接口的相反约定混淆。后续调用SDPA时显式设置推理 `dropout_p=0.0`，并核对融合后端的适用条件。

### **3. 执行方法、数据隔离与结课状态**

在项目根目录使用现有解释器执行本课；Jupyter内核需要本地socket权限，不需要模型网络访问。现有 `scripts/execute_learning.py` 可执行指定Notebook并导出HTML，目标文件就是本课；不要对S1–S3作覆盖重跑。直接阅读保存输出则不需要执行。

本课所有数值样本由固定种子的CPU generator在内存生成，不形成可混入正式训练的数据集。预览和审核记录集中在 `artifacts/learning/s8_01/`；它与 `data/`、正式表示缓存和工程run隔离。没有必要为了几组张量维护一份“真实数据集”。

| 验收层 | 状态与证据含义 |
| --- | --- |
| 教学数学与参考核 | 以本Notebook真实输出、独立复核及审核记录为准 |
| 个人核心与理解 | TODO待完成；不把看过答案记为通过 |
| 工程集成 | 本课未修改生产模型或训练入口 |
| 正式实验 | 未训练Chimera，未测CUDA吞吐／显存，未消费收费资源 |

S4–S8的学习材料在此形成闭环：你应能说明目标、表示、条件交互、采样、训练可靠性与研究对照，也能读懂新架构想节省的具体成本。课程材料交付与个人掌握、工程完成仍分别记录；接下来由真实主线实验检验这些判断。